# EDA 02 — Interim (preprocessed_from_raw) metadata file exploration

This notebook performs an exploratory analysis of the preprocessed raw metadata file after EDA_01 noteboook (file generated by *scripts/preprocess_data.py*).

The objective of this notebook is to explore the structure and content of the preprocessed metadata dataset. The analysis focuses on understanding the available candidate predictors for metadata-based machine learning models through descriptive statistics, missing value assessment, variable distributions, and basic exploratory analyses.

No feature engineering, feature selection, imputation, encoding, scaling, or other preprocessing steps are performed in this notebook.

In [1]:
# Libraries

import pandas as pd
import sys
from subprocess import run
from skin_lesion_ai.utils.data_utils import (
    get_project_root,
    load_metadata_parquet,
)

In [3]:
# Get latest 'preprocessed_from_raw' parquet file or generate it if it doesn't exist

repo_root = get_project_root()
script_path = repo_root / "scripts" / "preprocess_data.py"

try:
    df = load_metadata_parquet(
        stage="interim",
        filename="preprocessed_from_raw",
        timestamp_flag=True,
    )
except FileNotFoundError:
    run(
        [sys.executable, str(script_path)],
        cwd=str(repo_root),
        check=True,
    )
    df = load_metadata_parquet(
        stage="interim",
        filename="preprocessed_from_raw",
        timestamp_flag=True,
    )

In [4]:
# Preprocessed data

df

,isic_id,patient_id,attribution,copyright_license,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,tbp_lv_nevi_confidence,tbp_lv_dnn_lesion_confidence,tbp_lv_location,tbp_lv_location_simple,diagnostic_group
0,ISIC_0015670,IP_1235828,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,lower extremity,3.04,2.628592e-03,97.517282,Right Leg - Upper,Right Leg,benign_non_biopsied
1,ISIC_0015845,IP_8170065,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,head/neck,1.10,1.334303e-07,3.141455,Head & Neck,Head & Neck,benign_non_biopsied
2,ISIC_0015864,IP_6724798,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.40,2.959177e-04,99.804040,Torso Back Top Third,Torso Back,benign_non_biopsied
3,ISIC_0015902,IP_4111386,ACEMID MIA,CC-0,65.0,male,anterior torso,3.22,2.198945e+01,99.989998,Torso Front Top Half,Torso Front,benign_non_biopsied
4,ISIC_0024200,IP_8313778,Memorial Sloan Kettering Cancer Center,CC-BY,55.0,male,anterior torso,2.73,1.378832e-03,70.442510,Torso Front Top Half,Torso Front,benign_non_biopsied
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401054,ISIC_9999937,IP_1140263,"Department of Dermatology, Hospital Clínic de ...",CC-BY-NC,70.0,male,anterior torso,6.80,9.936233e+01,99.999988,Torso Front Top Half,Torso Front,benign_non_biopsied
401055,ISIC_9999951,IP_5678181,Memorial Sloan Kettering Cancer Center,CC-BY,60.0,male,posterior torso,3.11,2.311562e-01,99.999820,Torso Back Top Third,Torso Back,benign_non_biopsied
401056,ISIC_9999960,IP_0076153,"Frazer Institute, The University of Queensland...",CC-BY,65.0,female,anterior torso,2.05,5.994798e+01,99.999416,Torso Front Top Half,Torso Front,benign_non_biopsied
401057,ISIC_9999964,IP_5231513,University Hospital of Basel,CC-BY-NC,30.0,female,anterior torso,2.80,9.931933e+01,100.000000,Torso Front Bottom Half,Torso Front,benign_non_biopsied


# AGE

### 1) BASIC DESCRIPTION

### 2) MISSING VALUES

# SEX

### 1) BASIC DESCRIPTION

In [5]:
df["sex"].dtype

<StringDtype(na_value=nan)>

| Atributo                    | Valor                                              |
| --------------------------- | -------------------------------------------------- |
| Nombre de la variable       | `sex`                                              |
| Tipo de variable            | Cualitativa                                        |
| Subtipo                     | Binaria                                            |
| Tipo de almacenamiento      | `string`                                           |
| Valores observados          | `male`, `female`                                   |
| Presencia de *missing*      | Sí (`NA`)                                          |
| Nivel de observación        | Atributo del paciente almacenado a nivel de lesión |
| Codificación recomendada    | Binaria (`0/1`)                                    |


### 2) MISSING VALUES

In [8]:
# Missing sex analysis

# Lesion-level missing sex records
sex_missing_lesions = df.loc[df["sex"].isna()].copy()

# IDs of lesions and patients with missing sex
missing_sex_isic_ids = sex_missing_lesions["isic_id"].unique()
missing_sex_patient_ids = sex_missing_lesions["patient_id"].unique()

# Patient-level sex consistency
sex_per_patient = (
    df.groupby("patient_id")["sex"]
    .nunique(dropna=False)
    .reset_index(name="n_unique_sex_values")
)

patients_with_inconsistent_sex = sex_per_patient.query("n_unique_sex_values > 1")

# Missingness at lesion level
lesion_missing_summary = pd.DataFrame(
    {
        "level": ["Lesion-level"],
        "total_records": [len(df)],
        "missing_sex_records": [df["sex"].isna().sum()],
        "missing_sex_percentage": [round(100 * df["sex"].isna().mean(), 2)],
    }
)

# Missingness at patient level
patient_sex_summary = (
    df.groupby("patient_id")["sex"]
    .agg(
        sex_values=lambda x: sorted(x.dropna().unique()),
        has_missing_sex=lambda x: x.isna().any(),
        all_missing_sex=lambda x: x.isna().all(),
        n_lesions="size",
    )
    .reset_index()
)

patients_with_missing_sex = patient_sex_summary.query("has_missing_sex == True").copy()

patient_missing_summary = pd.DataFrame(
    {
        "level": ["Patient-level"],
        "total_records": [df["patient_id"].nunique()],
        "missing_sex_records": [patients_with_missing_sex["patient_id"].nunique()],
        "missing_sex_percentage": [
            round(
                100
                * patients_with_missing_sex["patient_id"].nunique()
                / df["patient_id"].nunique(),
                2,
            )
        ],
    }
)

# Combined summary
sex_missing_summary = pd.concat(
    [lesion_missing_summary, patient_missing_summary], ignore_index=True
)

sex_missing_summary

,level,total_records,missing_sex_records,missing_sex_percentage
0,Lesion-level,401059,11517,2.87
1,Patient-level,1042,33,3.17


| Nivel de análisis            |                  Total |       Registros con `sex` missing | % missing | Comentario                                                  |
| ---------------------------- | ---------------------: | --------------------------------: | --------: | ----------------------------------------------------------- |
| Lesión                       |                401,059 |                            11,517 |     2.88% | Corresponde a lesiones sin información de sexo del paciente |
| Paciente                     |                  1,042 |                                33 |     3.17% | Corresponde a pacientes sin sexo registrado                 |
| Consistencia paciente-lesión | 33 pacientes afectados | 33 con todas sus lesiones missing |   100.00% | La ausencia de `sex` es consistente dentro de cada paciente |

La variable `sex` presenta valores *missing* en 11,517 lesiones, equivalentes al 2.88% del total de registros a nivel de lesión. A nivel de paciente, estos valores corresponden a 33 pacientes de un total de 1,042, lo que representa un 3.17%.

Dado que `sex` es una característica propia del paciente y no de la lesión, se evaluó la consistencia de la ausencia de información dentro de cada paciente. En todos los pacientes afectados, la variable `sex` aparece como missing en todas sus lesiones, sin observarse casos de ausencia parcial. Por tanto, el patrón de missingness es coherente con la naturaleza patient-level de la variable.


# ANATOMICAL SITE

# LESION SIZE

# Final summary